#### Customer Churn Analysis

Use LAG to compare a customer’s current month activity with the previous month.

Example: Detect if a customer stopped transactions compared to last month → potential churn flag.

Customer Churn Analysis

Use LAG to compare a customer’s current month activity with the previous month.

Example: Detect if a customer stopped transactions compared to last month → potential churn flag.

Sales Trend Tracking

Use LAG to calculate month‑over‑month sales growth.

Example: (current_sales - lagged_sales) / lagged_sales gives growth percentage.

Employee Salary Progression

Use LAG to see an employee’s previous salary and LEAD to check the next scheduled increment.

Helps HR track pay progression and anomalies.

Stock Price Movement

Use LAG to get yesterday’s closing price and compare with today’s.

Use LEAD to forecast or prepare for next day’s expected movement.

Order Fulfillment Delays

Use LAG to calculate the time gap between consecutive orders for a customer.

Example: Identify customers with unusually long gaps between purchases.

Website Session Analysis

Use LEAD to find the next page a user visited after the current one.

Helps in clickstream analysis and building recommendation systems.

In [0]:
orders_data = [
    (1, 101, "2026-01-05", 250),
    (2, 101, "2026-01-15", 300),
    (3, 101, "2026-02-02", 150),
    (4, 102, "2026-01-07", 500),
    (5, 102, "2026-02-10", 450),
    (6, 102, "2026-03-01", 700),
    (7, 103, "2026-01-12", 200),
    (8, 103, "2026-01-25", 220),
    (9, 103, "2026-02-20", 180),
    (10, 104, "2026-01-18", 1000),
    (11, 104, "2026-03-05", 1200),
    (12, 105, "2026-02-01", 350),
    (13, 105, "2026-02-15", 400),
    (14, 105, "2026-03-10", 380),
    (15, 106, "2026-01-22", 600),
    (16, 106, "2026-02-25", 620),
    (17, 106, "2026-03-15", 580),
    (18, 107, "2026-01-30", 750),
    (19, 107, "2026-02-28", 800),
    (20, 107, "2026-03-20", 820)
]

columns = ['order_id','customer_id','order_date','order_amount']

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import *

df=spark.createDataFrame(orders_data,columns)

df.createOrReplaceTempView('orders')

In [0]:
%sql
select * from orders limit 5;

order_id,customer_id,order_date,order_amount
1,101,2026-01-05,250
2,101,2026-01-15,300
3,101,2026-02-02,150
4,102,2026-01-07,500
5,102,2026-02-10,450


#### Customer Purchase Gap Analysis

- Find the number of days between consecutive orders for each customer.
- Use LAG(order_date) to calculate gaps

In [0]:
%sql
with prev_orders_data as
(select *,
       lag(order_date) over(partition by customer_id order by order_date) as prev_order_date
from orders)

select order_id, customer_id ,order_date, order_amount, 
       datediff(order_date, prev_order_date) as days_gap
from prev_orders_data

order_id,customer_id,order_date,order_amount,days_gap
1,101,2026-01-05,250,null
2,101,2026-01-15,300,10
3,101,2026-02-02,150,18
4,102,2026-01-07,500,null
5,102,2026-02-10,450,34
6,102,2026-03-01,700,19
7,103,2026-01-12,200,null
8,103,2026-01-25,220,13
9,103,2026-02-20,180,26
10,104,2026-01-18,1000,null


In [0]:
## Pyspark code

from pyspark.sql import functions as F
from pyspark.sql import Window

# Define window partitioned by customer_id and ordered by order_date
window_spec = Window.partitionBy("customer_id").orderBy("order_date")

# Add previous order_date using LAG
prev_orders_data = df.withColumn(
    "prev_order_date",
    F.lag("order_date").over(window_spec)
)

# Calculate days gap
result_df = prev_orders_data.withColumn(
    "days_gap",
    F.datediff(F.col("order_date"), F.col("prev_order_date"))
)

# Show results
result_df.select("order_id", "customer_id", "order_date", "order_amount", "days_gap").show()


+--------+-----------+----------+------------+--------+
|order_id|customer_id|order_date|order_amount|days_gap|
+--------+-----------+----------+------------+--------+
|       1|        101|2026-01-05|         250|    NULL|
|       2|        101|2026-01-15|         300|      10|
|       3|        101|2026-02-02|         150|      18|
|       4|        102|2026-01-07|         500|    NULL|
|       5|        102|2026-02-10|         450|      34|
|       6|        102|2026-03-01|         700|      19|
|       7|        103|2026-01-12|         200|    NULL|
|       8|        103|2026-01-25|         220|      13|
|       9|        103|2026-02-20|         180|      26|
|      10|        104|2026-01-18|        1000|    NULL|
|      11|        104|2026-03-05|        1200|      46|
|      12|        105|2026-02-01|         350|    NULL|
|      13|        105|2026-02-15|         400|      14|
|      14|        105|2026-03-10|         380|      23|
|      15|        106|2026-01-22|         600|  

##### Sales Trend Detection

- Compare each month’s sales with the previous month to identify growth or decline.
- Use LAG(order_amount) for month‑over‑month comparison.

In [0]:
%sql
with month_sales as 
(select sum(order_amount) as month_sales, date_format(order_date, "yyyy-MM") as order_month from orders
group by date_format(order_date, "yyyy-MM")
order by order_month)

select *,
     case 
       when lag(month_sales) over(order by order_month) > month_sales then 'Declined'
       when lag(month_sales) over(order by order_month) < month_sales then 'Growth'
       else 'No Change'
     end as pattern_details
from month_sales

month_sales,order_month,pattern_details
3820,2026-01,No Change
2950,2026-02,Declined
3680,2026-03,Growth


In [0]:
## Pyspark code.. 

from pyspark.sql import functions as F
from pyspark.sql import Window

# Step 1: Extract year-month from order_date
orders_monthly = df.withColumn(
    "order_month",
    F.date_format(F.col("order_date"), "yyyy-MM")
)

# Step 2: Aggregate sales by month
monthly_sales = orders_monthly.groupBy("order_month").agg(
    F.sum("order_amount").alias("month_sales")
).orderBy("order_month")

# Step 3: Define window for month-over-month comparison
window_spec = Window.orderBy("order_month")

# Step 4: Add previous month’s sales using LAG
monthly_sales_with_lag = monthly_sales.withColumn(
    "prev_sales",
    F.lag("month_sales").over(window_spec)
)

# Step 5: Add categorical pattern (Growth / Declined / No Change)
result_df = monthly_sales_with_lag.withColumn(
    "pattern_details",
    F.when(F.col("prev_sales").isNull(), "No Change")
     .when(F.col("month_sales") > F.col("prev_sales"), "Growth")
     .when(F.col("month_sales") < F.col("prev_sales"), "Declined")
     .otherwise("No Change")
)

# Show results
result_df.select("order_month", "month_sales", "pattern_details").show()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-----------+---------------+
|order_month|month_sales|pattern_details|
+-----------+-----------+---------------+
|    2026-01|       3820|      No Change|
|    2026-02|       2950|       Declined|
|    2026-03|       3680|         Growth|
+-----------+-----------+---------------+



##### Next Purchase Prediction

For each order, show the next order date for the same customer.

Use LEAD(order_date) to anticipate customer buying cycles.

In [0]:
%sql
select
  *,
  lead(order_date) over (partition by customer_id order by order_date) as next_order_date
from
  orders

order_id,customer_id,order_date,order_amount,next_order_date
1,101,2026-01-05,250,2026-01-15
2,101,2026-01-15,300,2026-02-02
3,101,2026-02-02,150,null
4,102,2026-01-07,500,2026-02-10
5,102,2026-02-10,450,2026-03-01
6,102,2026-03-01,700,null
7,103,2026-01-12,200,2026-01-25
8,103,2026-01-25,220,2026-02-20
9,103,2026-02-20,180,null
10,104,2026-01-18,1000,2026-03-05


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

## Pyspark code

# Step 1: Define window partitioned by customer_id and ordered by order_date
window_spec = Window.partitionBy("customer_id").orderBy("order_date")

# Step 2: Add next_order_date using LEAD
orders_with_next = df.withColumn("next_order_date",F.lead("order_date").over(window_spec))

# Step 3: Select relevant columns for clarity
result_df = orders_with_next.select(
    "order_id",
    "customer_id",
    "order_date",
    "order_amount",
    "next_order_date")

# Step 4: Show the results
result_df.show()

+--------+-----------+----------+------------+---------------+
|order_id|customer_id|order_date|order_amount|next_order_date|
+--------+-----------+----------+------------+---------------+
|       1|        101|2026-01-05|         250|     2026-01-15|
|       2|        101|2026-01-15|         300|     2026-02-02|
|       3|        101|2026-02-02|         150|           NULL|
|       4|        102|2026-01-07|         500|     2026-02-10|
|       5|        102|2026-02-10|         450|     2026-03-01|
|       6|        102|2026-03-01|         700|           NULL|
|       7|        103|2026-01-12|         200|     2026-01-25|
|       8|        103|2026-01-25|         220|     2026-02-20|
|       9|        103|2026-02-20|         180|           NULL|
|      10|        104|2026-01-18|        1000|     2026-03-05|
|      11|        104|2026-03-05|        1200|           NULL|
|      12|        105|2026-02-01|         350|     2026-02-15|
|      13|        105|2026-02-15|         400|     2026

##### High‑Value Order Tracking

- Identify if a customer’s current order is higher or lower than their previous one.
- Use LAG(order_amount) to compare spending patterns.

In [0]:
%sql
-- Identify if a customer’s current order is higher or lower than their previous one.
-- Use LAG(order_amount) to compare spending patterns.
with current_prev_orders as 
(select *,
        lag(order_amount) over (partition by customer_id order by order_id)  as prev_amount 
      from orders)

select *, 
       case 
         WHEN prev_amount IS NULL THEN 'No Previous Order'
         when order_amount > prev_amount then 'Higher'
         when order_amount < prev_amount then 'Lower'
         else 'Same'
       end as spending_pattern
from current_prev_orders


order_id,customer_id,order_date,order_amount,prev_amount,spending_pattern
1,101,2026-01-05,250,null,No Previous Order
2,101,2026-01-15,300,250,Higher
3,101,2026-02-02,150,300,Lower
4,102,2026-01-07,500,null,No Previous Order
5,102,2026-02-10,450,500,Lower
6,102,2026-03-01,700,450,Higher
7,103,2026-01-12,200,null,No Previous Order
8,103,2026-01-25,220,200,Higher
9,103,2026-02-20,180,220,Lower
10,104,2026-01-18,1000,null,No Previous Order


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

## Pyspark code

# Define window partitioned by customer_id and ordered by order_amount
window_spec = Window.partitionBy("customer_id").orderBy("order_id")

# Step 1: Add previous order amount using LAG
current_prev_orders = df.withColumn(
    "prev_amount",
    F.lag("order_amount").over(window_spec)
)

# Step 2: Derive spending pattern
result_df = current_prev_orders.withColumn(
    "spending_pattern",
    F.when(F.col("prev_amount").isNull(), "No Previous Order")
     .when(F.col("order_amount") > F.col("prev_amount"), "Higher")
     .when(F.col("order_amount") < F.col("prev_amount"), "Lower")
     .otherwise("Same")
)

# Show results
result_df.select("order_id", "customer_id", "order_amount", "prev_amount", "spending_pattern").show()

+--------+-----------+------------+-----------+-----------------+
|order_id|customer_id|order_amount|prev_amount| spending_pattern|
+--------+-----------+------------+-----------+-----------------+
|       1|        101|         250|       NULL|No Previous Order|
|       2|        101|         300|        250|           Higher|
|       3|        101|         150|        300|            Lower|
|       4|        102|         500|       NULL|No Previous Order|
|       5|        102|         450|        500|            Lower|
|       6|        102|         700|        450|           Higher|
|       7|        103|         200|       NULL|No Previous Order|
|       8|        103|         220|        200|           Higher|
|       9|        103|         180|        220|            Lower|
|      10|        104|        1000|       NULL|No Previous Order|
|      11|        104|        1200|       1000|           Higher|
|      12|        105|         350|       NULL|No Previous Order|
|      13|

##### Customer Retention Flag

- Mark customers who haven’t placed an order after their last purchase.
- Use LEAD(order_date) → if NULL, it’s their last order.

In [0]:
%sql
with transactions_details as 
(select *,
       lead(order_date) over (partition by customer_id order by order_date) as next_order_date
 from orders)

 select *, 
        case 
            when next_order_date is null then 'Last order'
            else 'Active'
        end as last_order_status
 from transactions_details

order_id,customer_id,order_date,order_amount,next_order_date,last_order_status
1,101,2026-01-05,250,2026-01-15,Active
2,101,2026-01-15,300,2026-02-02,Active
3,101,2026-02-02,150,null,Last order
4,102,2026-01-07,500,2026-02-10,Active
5,102,2026-02-10,450,2026-03-01,Active
6,102,2026-03-01,700,null,Last order
7,103,2026-01-12,200,2026-01-25,Active
8,103,2026-01-25,220,2026-02-20,Active
9,103,2026-02-20,180,null,Last order
10,104,2026-01-18,1000,2026-03-05,Active


In [0]:
## Pyspark code
 
from pyspark.sql import functions as F
from pyspark.sql import Window

# Step 1: Define window partitioned by customer_id and ordered by order_date
window_spec = Window.partitionBy("customer_id").orderBy("order_date")

# Step 2: Add next_order_date using LEAD
transactions_details = df.withColumn(
    "next_order_date",
    F.lead("order_date").over(window_spec)
)

# Step 3: Add last_order_status flag
transactions_flagged = transactions_details.withColumn(
    "last_order_status",
    F.when(F.col("next_order_date").isNull(), "Last order")
     .otherwise("Not last order")
)

# Step 4: Select relevant columns for clean output
result_df = transactions_flagged.select(
    "order_id",
    "customer_id",
    "order_date",
    "order_amount",
    "next_order_date",
    "last_order_status"
)

# Step 5: Show results
result_df.show()

+--------+-----------+----------+------------+---------------+-----------------+
|order_id|customer_id|order_date|order_amount|next_order_date|last_order_status|
+--------+-----------+----------+------------+---------------+-----------------+
|       1|        101|2026-01-05|         250|     2026-01-15|   Not last order|
|       2|        101|2026-01-15|         300|     2026-02-02|   Not last order|
|       3|        101|2026-02-02|         150|           NULL|       Last order|
|       4|        102|2026-01-07|         500|     2026-02-10|   Not last order|
|       5|        102|2026-02-10|         450|     2026-03-01|   Not last order|
|       6|        102|2026-03-01|         700|           NULL|       Last order|
|       7|        103|2026-01-12|         200|     2026-01-25|   Not last order|
|       8|        103|2026-01-25|         220|     2026-02-20|   Not last order|
|       9|        103|2026-02-20|         180|           NULL|       Last order|
|      10|        104|2026-0


##### Revenue Spike Detection

- Detect sudden jumps in order amounts compared to the previous order.
- Use LAG(order_amount) and calculate percentage change.

##### First vs Last Order Analysis

- For each customer, show their first order date and last order date.
- Use LAG and LEAD with ROW_NUMBER() or RANK() to identify boundaries.